# Step 5: Stepwise selection creates new models

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** Step 2 (`step2_terms.json`). **Saves:** `data/processed/model_selection/stepwise.json`, used by Step 6.
This is the slow step (about 5–8 minutes): only rerun it when Step 2 or the stepwise rules change.

Stepwise selection builds a series of models by adding one term at a time: at each step, the term that improves
**BIC** the most. BIC rewards a better fit but penalises every extra parameter. It stops when no term improves BIC.
Terms can also be removed if that improves BIC.

- **Forward:** starts from an empty model; every step is saved as a new model (S1, S2, ...).
- **Backward:** starts from the full model and removes terms while BIC improves.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
full_terms = mu.read("step2_terms", made_by="step2_collinearity.ipynb")

bic_cache = {}
def bic(terms):
    key = frozenset(terms)
    if key not in bic_cache:
        bic_cache[key] = mu.ols(terms).bic
    return bic_cache[key]

def stepwise(pool, start):
    current, path = list(start), []
    best = bic(current)
    while True:
        moves = [current + [t] for t in pool if t not in current] + [[c for c in current if c != t] for t in current]
        moves = [m for m in moves if mu.with_hierarchy(m) == m]
        if not moves:
            break
        nxt = min(moves, key=bic)
        if bic(nxt) >= best:
            break
        added, removed = set(nxt) - set(current), set(current) - set(nxt)
        change = f"+ {added.pop()}" if added else f"- {removed.pop()}"
        current, best = nxt, bic(nxt)
        path.append({"change": change, "terms": list(current), "BIC": best})
    return path

forward = stepwise(full_terms, [])
backward = stepwise(full_terms, full_terms)
mu.save("stepwise", {"forward": forward, "backward": backward})
for name, path in [("forward", forward), ("backward", backward)]:
    print(f"{name}: {len(path)} steps")
    for i, p in enumerate(path, 1):
        print(f"  {i:>2}. {p['change']:<22} BIC = {p['BIC']:,.0f}")

**Results.**
- **Forward:** adds the terms in this order: last week's demand, yesterday's demand, day of week, hour, the weekend
  pattern, zone, zone × weekend, holiday, month, rain, sunlight, cloud cover. That gives 12 new models, **S1 to S12**.
  It stops there: adding new snow or snow on the ground doesn't improve BIC.
- **Backward:** starts from the full model and removes the two snow terms, ending at the same model as S12.

So stepwise agrees with the F-tests on snow (no measurable effect). BIC keeps sunlight and cloud cover even though
they aren't significant in Step 3: BIC treats all 345k rows as independent, so small improvements count, while the
F-tests correctly treat hours on the same day as related.